In [1]:
from pathlib import Path

In [2]:
import gitinfo
import numpy as np
import pandas as pd
import strictyaml
import xarray as xr
from tqdm import tqdm

In [3]:
from data import ds_single, ds_pressure

In [4]:
expert = pd.read_csv("expert.csv")

In [5]:
def parse_float(x: str):
    if len(x.strip()) == 0:
        return np.nan
    return float(x.replace(",", "."))

In [6]:
recommendations = Path("compression-recommendations") / "recommendations"

In [7]:
def initialise_recommendation(
    key: str, grib_key: str, level: str, expert, da: xr.DataArray
) -> None:
    filters = [
        {
            "kind": "any",
            "comment": "variables",
            "filters": [
                {
                    "kind": "any",
                    "comment": da.long_name,
                    "filters": (
                        [
                            {
                                "kind": "cf-standard-name",
                                "value": da.standard_name,
                            }
                        ]
                        if da.standard_name.lower() != "unknown"
                        else []
                    )
                    + [
                        {
                            "kind": "cf-short-name",
                            "value": key,
                        },
                        {
                            "kind": "grib-short-name",
                            "value": grib_key,
                        },
                    ],
                },
            ],
        },
        {
            "kind": "level-kind",
            "value": level,
        }
    ]

    absl1 = parse_float(str(expert["Absolute L1 error"]))
    abslinf = parse_float(str(expert["Absolute Linfty error"]))
    rell1 = parse_float(str(expert["Relative L1 error"]))
    rellinf = parse_float(str(expert["Relative Linfty error"]))

    abs_requirements = []
    rel_requirements = []

    if not np.isnan(absl1):
        abs_requirements.append(
            {
                "kind": "mean-pointwise-absolute-error-bound",
                "value": absl1,
            }
        )
    if not np.isnan(abslinf):
        abs_requirements.append(
            {
                "kind": "max-pointwise-absolute-error-bound",
                "value": abslinf,
            }
        )
    if not np.isnan(rell1):
        rel_requirements.append(
            {
                "kind": "mean-pointwise-relative-error-bound",
                "value": rell1,
            }
        )
    if not np.isnan(rellinf):
        rel_requirements.append(
            {
                "kind": "max-pointwise-relative-error-bound",
                "value": rellinf,
            }
        )

    if len(abs_requirements) == 0 and len(rel_requirements) == 0:
        return

    requirements = [
        {
            "kind": "any",
            "comment": "pointwise error bounds",
            **(
                {
                    "comment-2": "",
                    "comment-ml-start": True,
                    "_": expert["Comments"].strip(),
                    "comment-ml-end": True,
                }
                if isinstance(expert["Comments"], str)
                and len(expert["Comments"].strip()) > 0
                else {}
            ),
            "comment-3": "",
            "comment-4": "GRIB adds in ERA5 a maximum pointwise absolute error of",
            "comment-5": f"{parse_float(expert['Existing L1 error'])} [{expert['Unit']}]",
            "requirements": (
                [
                    {
                        "kind": "all",
                        "comment": "absolute error",
                        "requirements": abs_requirements,
                    }
                ]
                if len(abs_requirements) > 0
                else []
            )
            + (
                [
                    {
                        "kind": "all",
                        "comment": "relative error (0.01 = 1%)",
                        "requirements": rel_requirements,
                    }
                ]
                if len(rel_requirements) > 0
                else []
            ),
        }
    ]

    yaml = f"""\
# Requirements for {da.long_name}
#
# Initialised by @juntyr and @SF-N.
#
# Please feel free to add comments (with your name),
# and make well-justified changes!

""" + strictyaml.as_document(
        {
            "filters": filters,
            "requirements": requirements,
        }
    ).as_yaml()

    yaml2 = []
    is_ml_comment = False
    for line in yaml.splitlines(keepends=True):
        if is_ml_comment:
            if "comment-ml-end" in line:
                is_ml_comment = False
                continue
            else:
                for marker in ["_:"]:
                    x = line.find(marker)
                    if x != -1:
                        yaml2.append((" " * x) + "#" + line[x + 2 :])
                        break
                else:
                    yaml2.append(
                        (" " * (len(line) - len(line.lstrip(" ")) - 2))
                        + "# "
                        + line.lstrip(" ")
                    )
                continue
        elif "comment-ml-start" in line:
            is_ml_comment = True
            continue
        for marker in ["comment:"] + [f"comment-{i}:" for i in range(2, 5 + 1)]:
            x = line.find(marker)
            if x != -1:
                line = (" " * x) + "#" + line[x + len(marker) :]
                break
        yaml2.append(line)
    yaml = "".join(yaml2)

    assert strictyaml.load(yaml).as_yaml() == yaml, key

    recommendations.joinpath(f"{level}-{key}.yaml").write_text(yaml)

In [8]:
for key in tqdm(sorted(ds_single.mean)):
    grib_key = ds_single.mean[key].GRIB_shortName

    ex = expert[
        (expert["Short Name"] == grib_key) & (expert["Level"] == "Single/Surface")
    ]
    if len(ex) != 1:
        continue
    ex = ex.iloc[0]

    initialise_recommendation(key, grib_key, "single", ex, ds_single.mean[key])

100%|███████████████████████████████████████████████████████████████████████| 226/226 [00:01<00:00, 116.11it/s]


In [9]:
for key in tqdm(sorted(ds_pressure.mean)):
    grib_key = ds_pressure.mean[key].GRIB_shortName

    ex = expert[
        (expert["Short Name"] == grib_key) & (expert["Level"] == "Pressure+Model")
    ]
    if len(ex) != 1:
        continue
    ex = ex.iloc[0]

    initialise_recommendation(key, grib_key, "pressure", ex, ds_pressure.mean[key])

100%|██████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 56.80it/s]
